## Transfer Learning, обучение

ResNet50 (ImageNet V2, ~25.6M параметров) + новый `fc(2048→102)`. Две стадии — warmup головы (5 эпох, lr=1e-3) и fine-tuning `layer4`+`fc` (10 эпох, lr=1e-4 / 1e-5, CosineAnnealing). Loss — CrossEntropy.

In [ ]:
# !pip install torch torchvision scikit-learn tqdm matplotlib

import json
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, ConcatDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.__version__)


## Данные

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# train (1020) + val (1020) для обучения, test (6149) — итоговая оценка
train_part = torchvision.datasets.Flowers102(root='./data', split='train',
                                             download=True, transform=train_transform)
val_part   = torchvision.datasets.Flowers102(root='./data', split='val',
                                             download=True, transform=train_transform)
trainset   = ConcatDataset([train_part, val_part])

testset    = torchvision.datasets.Flowers102(root='./data', split='test',
                                             download=True, transform=eval_transform)

NUM_CLASSES = 102
BATCH_SIZE  = 32

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)
testloader  = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'train+val: {len(trainset)} | test: {len(testset)} | классов: {NUM_CLASSES}')


In [ ]:
peek_ds = torchvision.datasets.Flowers102(root='./data', split='train', download=True,
                                          transform=transforms.Compose([
                                              transforms.Resize(256),
                                              transforms.CenterCrop(224),
                                              transforms.ToTensor(),
                                          ]))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, (img, lbl) in zip(axes.ravel(), [peek_ds[i] for i in range(10)]):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f'class {lbl}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## Модель

In [ ]:
# ResNet50 = 50 «весовых» слоёв: 1 нач. conv + bottleneck-блоки
# layer1 (3 блока, 256 каналов), layer2 (4, 512), layer3 (6, 1024), layer4 (3, 2048).
# Меняем последний fc (2048) на 2048 -> 102.
def build_model(num_classes=NUM_CLASSES):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def set_trainable(model, train_layer4=False, train_fc=True):
    for p in model.parameters():
        p.requires_grad = False
    if train_layer4:
        for p in model.layer4.parameters():
            p.requires_grad = True
    if train_fc:
        for p in model.fc.parameters():
            p.requires_grad = True


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = build_model().to(device)
set_trainable(model, train_layer4=False, train_fc=True)
print(f'всего:    {sum(p.numel() for p in model.parameters()):,}')
print(f'stage 1:  {count_trainable(model):,}')


## Обучение

In [ ]:
loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss_sum += loss_fn(logits, y).item() * X.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += X.size(0)
    return loss_sum / total, correct / total


def run_stage(model, optimizer, scheduler, num_epochs, history, stage_name,
              ckpt_path='best_resnet50_flowers.pt', best_acc=0.0):
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0
        t0 = time.time()
        for X, y in trainloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            running_loss    += loss.item() * X.size(0)
            running_correct += (logits.argmax(1) == y).sum().item()
            running_total   += X.size(0)
        if scheduler is not None:
            scheduler.step()

        train_loss = running_loss / running_total
        train_acc  = running_correct / running_total
        test_loss, test_acc = evaluate(model, testloader)

        history['stage'].append(stage_name)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), ckpt_path)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'[{stage_name}] epoch {epoch:02d}/{num_epochs} | '
              f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
              f'test loss {test_loss:.4f} acc {test_acc:.4f} | '
              f'lr {lr_now:.6f} | {time.time()-t0:.1f}s')
    return best_acc


history = {'stage': [], 'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc = 0.0


### Стадия 1: warmup головы

In [ ]:
# Стадия 1: только fc обучается, backbone заморожен
STAGE1_EPOCHS = 5
STAGE1_LR     = 1e-3

set_trainable(model, train_layer4=False, train_fc=True)
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=STAGE1_LR, weight_decay=1e-4,
)
scheduler = None

best_acc = run_stage(model, optimizer, scheduler, STAGE1_EPOCHS, history, 'warmup',
                     best_acc=best_acc)
print(f'после warmup: {best_acc:.4f}')


### Стадия 2: fine-tuning `layer4` + `fc`

In [ ]:
# Стадия 2: размораживаем layer4 + fc. lr головы x10 больше, чтобы не разрушить backbone
STAGE2_EPOCHS = 10
STAGE2_LR_HEAD     = 1e-4
STAGE2_LR_BACKBONE = 1e-5

set_trainable(model, train_layer4=True, train_fc=True)
optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': STAGE2_LR_BACKBONE},
    {'params': model.fc.parameters(),     'lr': STAGE2_LR_HEAD},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS)

print(f'stage 2: {count_trainable(model):,}')

best_acc = run_stage(model, optimizer, scheduler, STAGE2_EPOCHS, history, 'finetune',
                     best_acc=best_acc)
print(f'best test acc: {best_acc:.4f}')


## Сохранение

In [ ]:
with open('history.json', 'w') as f:
    json.dump(history, f, indent=2)

# в Colab:
# from google.colab import files
# files.download('best_resnet50_flowers.pt')
# files.download('history.json')


In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
switch = history['stage'].index('finetune') + 1 if 'finetune' in history['stage'] else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['test_loss'],  label='test')
axes[0].set_title('loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, history['train_acc'], label='train')
axes[1].plot(epochs, history['test_acc'],  label='test')
axes[1].set_title('accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
if switch is not None:
    for ax in axes:
        ax.axvline(switch - 0.5, color='gray', linestyle='--', alpha=0.7)
plt.tight_layout(); plt.show()
